# Token generation & perplexity on dish names

**Question.** When a model reads a culinary sentence, is the *dish name* the hard part?

**Method.** Pseudo-perplexity: the sentence is the target, the decoder is causal, so token *t* is scored
given tokens *< t*. Every token is tagged **dish** or **other**; the comparison is paired within the sentence.

In [ ]:
import csv, json, os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name != "research-prototype" else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from src import config as C, models as M, data as D

def read_csv(name):
    p = ROOT / "results" / name
    if not p.exists():
        print(f"missing {p.name} - run: python -m src.run_all (stage not executed yet)")
        return []
    return list(csv.DictReader(p.open(encoding="utf-8")))

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 1. The test set — 100 authored pairs

In [ ]:
pairs = D.build(force=False)
for r in pairs[:5]:
    print(f"[{r['id']:3d}] {r['dish_vi']:<22} | {r['vi'][:70]}")
    print(f"      {r['dish_en']:<22} | {r['en'][:70]}")

## 2. Live — per-token NLL for one sentence

In [ ]:
from src import score

tok, model, _ = M.load("vit5-base")
rows = score.explain(tok, model, pairs[4]["vi"], pairs[4]["dish_vi"])
print(f"{'pos':>4} {'piece':<14}{'span':<16}{'dish':<6}{'nll':>7}{'rank':>7}")
for r in rows:
    print(f"{r['pos']:>4} {r['piece']:<14}{r['span']:<16}{str(r['dish']):<6}{r['nll']:>7.2f}{r['rank']:>7}")

joined = "".join(r["span"] for r in rows if r["dish"]).strip()
assert joined.lower() == pairs[4]["dish_vi"].lower(), (joined, pairs[4]["dish_vi"])
print(f"\nOK: dish-token spans concatenate back to {joined!r}")
M.free(model)

`rank` = position of the true token in the model's own distribution. Rank 1 means the model would have produced it. This is the most human-readable version of the same quantity as NLL.

## 3. Aggregate — dish vs the rest

In [ ]:
table = read_csv("ppl_summary.csv")
cols = ["model", "lang", "ppl_all", "ppl_dish", "rank_dish", "ppl_other", "rank_other", "delta_nll", "delta_ci_lo", "delta_ci_hi", "dish_harder_sentences", "sentences"]
if table:
    print(" | ".join(cols))
    for r in table:
        print(" | ".join(str(r[c]) for c in cols))

## 4. How many sentences show the effect?

In [ ]:
import matplotlib.pyplot as plt

for r in read_csv("ppl_summary.csv"):
    lo, hi, d = float(r["delta_ci_lo"]), float(r["delta_ci_hi"]), float(r["delta_nll"])
    sig = "significant" if (lo > 0 or hi < 0) else "not significant"
    print(f"{r['model']:<14} delta {d:+.3f} nats  CI [{lo:+.3f}, {hi:+.3f}]  {sig}  "
          f"({r['dish_harder_sentences']}/{r['sentences']} sentences)")

## 5. Caveats that decide how strong the claim can be

- This is a **copy loss**, not a corpus language-model perplexity: the encoder sees the whole sentence, so
  the model is scored on reproducing tokens it has already read. Comparable **within** a model only.
- A dish name is often a rare proper-ish noun, so some elevation is expected even without any culinary
  specialness. The honest control is the within-sentence comparison, plus the `rank` column.
- Vietnamese and English numbers are not on the same scale (different vocabularies, different tokenisation
  density: a Vietnamese dish name can be 1 piece, an English transliteration 6).
- Position matters: the first token has no decoder context. It is kept in the CSV (`has_context`) and
  excluded from the summary.
